# 🚗 TÁI HIỆN BASELINE: FOCALCOMM GỐC CHÍNH THỨC (WACV 2026)
### 📘 Hard Instance-Aware Multi-Agent Perception (HIM + QAFF)

Notebook này dùng để **huấn luyện và đánh giá mô hình Baseline chuẩn của đề tài**:
👉 **Mô hình FocalComm gốc chính thức được công bố tại WACV 2026** (sử dụng cơ chế trao đổi đặc trưng cộng tác HIM + QAFF, đạt mốc chuẩn **56.1% mAP** trên tập V2X-Real).

Mã nguồn được clone trực tiếp từ repository: [tranvuquanghuy12/FocalComm](https://github.com/tranvuquanghuy12/FocalComm.git).

---
### 📌 Hai quy trình thực thi chính:
1. **Quy trình 1 (Nghiệm thu Baseline tức thì):** Tải trọng số Pretrained Checkpoint chính thức của tác giả FocalComm và chạy Inference để kiểm chứng ngay con số **56.1% mAP** (Table 1 của bài báo).
2. **Quy trình 2 (Huấn luyện lại Baseline từ đầu):** Khởi chạy huấn luyện toàn diện mô hình FocalComm gốc (HIM + QAFF) trên Kaggle GPU.

## 1. Kiểm tra phần cứng & GPU trên Kaggle
Yêu cầu: GPU Tesla T4 (hoặc P100) và chế độ Internet: ON.

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: CUDA chưa bật! Vui lòng chọn Accelerator = GPU trong cài đặt bên phải Kaggle.")

!nvidia-smi


## 2. Clone Repository FocalComm từ GitHub
Lấy toàn bộ mã nguồn chuẩn của mô hình FocalComm gốc.

In [ ]:
# Xóa thư mục cũ nếu có để đảm bảo lấy code mới nhất
!rm -rf /kaggle/working/FocalComm
!git clone https://github.com/tranvuquanghuy12/FocalComm.git /kaggle/working/FocalComm

%cd /kaggle/working/FocalComm
!git log -n 1 --oneline
!ls -la


## 3. Cài đặt các thư viện & Biên dịch CUDA Extensions
Mô hình FocalComm gốc sử dụng thư viện thưa , OpenPCDet CUDA Ops () và Cython ().

In [ ]:
# 1. Cài đặt các packages cần thiết
!pip install -q easydict numba open3d cython tensorboardX shapely==2.0.0
!pip install -q pyquaternion einops timm icecream python-lzf termcolor efficientnet_pytorch==0.7.0 huggingface_hub
!pip install -q git+https://github.com/klintan/pypcd.git

# 2. Cài đặt spconv phù hợp với CUDA Kaggle
!pip install -q spconv-cu118 || pip install -q spconv-cu120 || pip install -q spconv-cu117

# 3. Biên dịch CUDA & Cython Extensions của dự án
%cd /kaggle/working/FocalComm/focalcomm/utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm/focalcomm/pcdet_utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm
print("✅ Môi trường và CUDA Extensions đã sẵn sàng!")


## 4. Thiết lập Bộ Dữ Liệu V2X-Real
Tạo liên kết dữ liệu V2X-Real từ Kaggle input.

In [ ]:
import os
import shutil

DATASET_ROOT = "/kaggle/working/dataset/V2XReal"
os.makedirs(DATASET_ROOT, exist_ok=True)

# Tự động phát hiện dữ liệu nếu đã add dataset trên Kaggle
kaggle_input_dirs = [d for d in os.listdir("/kaggle/input") if "v2x" in d.lower() or "focalcomm" in d.lower()] if os.path.exists("/kaggle/input") else []
if kaggle_input_dirs:
    src = os.path.join("/kaggle/input", kaggle_input_dirs[0])
    print(f"Phát hiện dataset trong Kaggle Input: {src}")
    if not os.path.exists(os.path.join(DATASET_ROOT, "train")) and os.path.exists(os.path.join(src, "train")):
        os.symlink(os.path.join(src, "train"), os.path.join(DATASET_ROOT, "train"))
    if not os.path.exists(os.path.join(DATASET_ROOT, "val")) and os.path.exists(os.path.join(src, "val")):
        os.symlink(os.path.join(src, "val"), os.path.join(DATASET_ROOT, "val"))
    print(f"Liên kết dữ liệu thành công vào: {DATASET_ROOT}")
else:
    print(f"Dataset sẽ được đọc từ: {DATASET_ROOT}")


## 5. QUY TRÌNH 1: Đánh Giá Nghiệm Thu Baseline FocalComm Gốc (56.1% mAP)
Tải trực tiếp trọng số Checkpoint chính thức của tác giả FocalComm từ Hugging Face và chạy kiểm thử trên tập V2X-Real.
👉 **Kết quả đối chứng chuẩn của bài báo: mAP@0.3 = 56.1%, AP@0.5 = 45.8%.**

In [ ]:
from huggingface_hub import snapshot_download
import os

ckpt_dir = "checkpoints/focalcomm_v2xreal"
os.makedirs(ckpt_dir, exist_ok=True)

print("Đang tải checkpoint chính thức của FocalComm gốc từ Hugging Face (scdrand23/FocalComm)...")
try:
    snapshot_download(
        repo_id="scdrand23/FocalComm",
        local_dir="checkpoints",
        allow_patterns=["*focalcomm_v2xreal*", "*net_epoch_best.pth*"]
    )
    print("✅ Đã tải checkpoint FocalComm gốc thành công!")
except Exception as e:
    print(f"Thông báo: {e}")

# Chạy đánh giá mô hình Baseline FocalComm gốc trên tập Val/Test
!python focalcomm/tools/inference.py \n    --model_dir checkpoints/focalcomm_v2xreal \n    --fusion_method intermediate


## 6. QUY TRÌNH 2: Huấn Luyện Lại Baseline FocalComm Gốc (HIM + QAFF)
Huấn luyện lại từ đầu mô hình FocalComm v3 chính thức với file cấu hình YAML đã được tối ưu hóa VRAM cho Kaggle GPU.

In [ ]:
# Huấn luyện FocalComm gốc với cơ chế cộng tác HIM + QAFF
!python focalcomm/tools/train_focalcomm.py \n    --hypes_yaml focalcomm/hypes_yaml/v2xreal/focalcommv3_kaggle.yaml


## 7. Đóng Gói Checkpoints và Kết Quả Tải Về Máy
Nén toàn bộ checkpoints huấn luyện được vào file  để tải về.

In [ ]:
output_tar = "/kaggle/working/focalcomm_baseline_results.tar.gz"
!tar -czvf {output_tar} checkpoints/
print(f"✅ Đã tạo file kết quả tại: {output_tar}")
